After training the attentive classifier with verb, manipulated and affected prediction objective. We don't need the classes output logits, we only need the feature probing from attentive pooler. Hence the final feature is the output feature coming from attentive pooler.

# Load classifier with weights

In [1]:
import yaml
import json
from pprint import pprint

from src.classifiers import init_classifier

# load num classes
with open('data/annotations/annotation_all.json', 'r') as f:
    annota = json.load(f)
typeinfo = annota['type_info']
NUM_CLASSES = {k: v['num_classes'] for k,v in typeinfo.items()}
print(NUM_CLASSES)
# load config
with open('configs/feature_extraction_vjepa2.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
cls_cfg = cfg['classifier']
assert cls_cfg is not None, 'Classifier config is emtpy!'

pprint(cfg)

classifer = init_classifier(
    checkpoint='ckpt/epoch1.pt',
    num_verb_classes=NUM_CLASSES['verb'],
    num_mani_classes=NUM_CLASSES['manipulated'],
    num_affect_classes=NUM_CLASSES['affected'],
    **cls_cfg
)


/home/tomtom/miniconda3/envs/k-pro/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


{'verb': 10, 'manipulated': 35, 'affected': 36, 'atomic_operation': 244, 'hand': 2}
{'classifier': {'embed_dim': 1024, 'num_blocks': 4, 'num_heads': 16},
 'decode_kwargs': {'clip_stride': 4, 'frames_per_clip': 16, 'sampling_rate': 1},
 'video_dir': 'data/FineBio/03 finebio_videos_fpv_all_w640'}


In [2]:
# for p in classifer.parameters():
#     print(p.device)
classifer.pooler

AttentivePooler(
  (cross_attention_block): CrossAttentionBlock(
    (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (xattn): CrossAttention(
      (q): Linear(in_features=1024, out_features=1024, bias=True)
      (kv): Linear(in_features=1024, out_features=2048, bias=True)
    )
    (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (mlp): MLP(
      (fc1): Linear(in_features=1024, out_features=4096, bias=True)
      (act): GELU(approximate='none')
      (fc2): Linear(in_features=4096, out_features=1024, bias=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
  )
  (blocks): ModuleList(
    (0-2): 3 x Block(
      (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
   

# Load V-JEPA 2

In [3]:
from src.models import init_vjepa2

model, processor = init_vjepa2()

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

model total parameters: 325.97M


# Extract loop

In [4]:
import os
import yaml
import torch
import numpy as np
from pprint import pprint
from tqdm import tqdm

from src.data_utils import get_video_paths, decode_video_to_clips

In [5]:
# === prepare save dir ===
save_dir = 'feats'
if not os.path.isdir(save_dir):
    os.mkdir(save_dir)

# === save config ===
with open(os.path.join(save_dir, "feature_extraction_config.txt"), "w") as f:
    yaml.dump(cfg, f)

# === init data ===
video_path_list = get_video_paths(
    data_dir=cfg["video_dir"], 
    # parameters for resuming feature extraction
    target_view_id="T0", 
    min_par_id=18
)

In [ ]:
import itertools

# === iterate videos ===
device = model.device
decode_cfg = cfg['decode_kwargs']
batch_size = 64

classifer.eval()
model.eval()

def clip_loader(iterable, batch_size):
    iterator = iter(iterable)
    while True:
        batch = list(itertools.islice(iterator, batch_size))
        if not batch:
            break
        yield batch
        
for path in tqdm(video_path_list):
    clips = decode_video_to_clips(
        video_path=path,
        frames_per_clip=decode_cfg["frames_per_clip"],
        clip_stride=decode_cfg["clip_stride"],
        sampling_rate=decode_cfg["sampling_rate"],
        num_threads=32
    )

    total_feat_list = []
    num_seen_clips = 0
    expected_num_clips = None
    
    # === process #batch_size clips together ===
    for batch_clips in clip_loader(clips, batch_size): 
        if expected_num_clips is None and "num_clips" in batch_clips[0]:
            expected_num_clips = batch_clips[0]["num_clips"]

        buffers = [clip["buffer"].cuda() for clip in batch_clips]
        inputs = processor(
            buffers,
            return_tensors="pt",
            do_resize=False
        )["pixel_values_videos"].float()

        with torch.no_grad():
            feat = model(inputs).last_hidden_state # ['last_hidden_state', 'masked_hidden_state', 'predictor_output']
            feat = classifer.pooler(feat) # B, 3, D
        
        total_feat_list.append(feat.detach().cpu().numpy())
        num_seen_clips += feat.shape[0] 
    
    if len(total_feat_list) == 0:
        print(f"Warning: no clips decoded for {path}")
        continue
    
    total_feat_list = np.concatenate(total_feat_list, axis=0) # number of clips, 3, D
    
    # final check before save to ensure actionformer will not fail due to mismatch in temporal stride
    assert total_feat_list.shape[0] == num_seen_clips, (
        f"Feature count mismatch: feat_array has {total_feat_list.shape[0]}, "
        f"but counted {num_seen_clips} processed clips"
    )
    if expected_num_clips is not None:
        assert total_feat_list.shape[0] == expected_num_clips, (
            f"Invalid feature length! got {total_feat_list.shape[0]} "
            f"expected {expected_num_clips}"
        )

    # === save video feat ===
    video_id = os.path.splitext(os.path.basename(path))[0]
    feat_path = video_id + ".npy"
    np.save(os.path.join(save_dir, feat_path), total_feat_list)
    print(f"{video_id}: num feat = ", total_feat_list.shape[0])

  0%|          | 0/100 [00:00<?, ?it/s]/home/tomtom/miniconda3/envs/k-pro/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
  1%|          | 1/100 [03:03<5:02:41, 183.45s/it]

P18_01_01: num feat =  1174
